In [ ]:
import os
import requests

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-04.parquet"
file_path = "../data/yellow_tripdata_2026-04.parquet"

# Baixa o arquivo apenas se ele não existir
if not os.path.exists(file_path):
    print("Baixando dataset...")
    response = requests.get(url)
    with open(file_path, "wb") as f:
        f.write(response.content)
    print("Download concluído!")
else:
    print("Arquivo já existe na pasta data/.")

In [ ]:
import pandas as pd
import duckdb
import time

file_path = "../data/raw/yellow_tripdata_2026-01.parquet"

print("Iniciando o teste de performance...\n")

# ==========================================
# 1. ABORDAGEM COM PANDAS
# ==========================================
start_pandas = time.time()

# O Pandas precisa carregar o arquivo INTEIRO para a memória RAM primeiro
df_pandas = pd.read_parquet(file_path)

# Só depois ele faz o filtro e a agregação
df_pandas['data_corrida'] = df_pandas['tpep_pickup_datetime'].dt.date
resultado_pandas = (
    df_pandas[
        (df_pandas['tpep_pickup_datetime'] >= '2026-04-01') & 
        (df_pandas['tpep_pickup_datetime'] < '2026-05-01')
    ]
    .groupby('data_corrida')
    .agg(
        total_corridas=('tpep_pickup_datetime', 'count'),
        ticket_medio=('fare_amount', 'mean')
    )
)

tempo_pandas = time.time() - start_pandas
print(f"Tempo Pandas: {tempo_pandas:.4f} segundos")


# ==========================================
# 2. ABORDAGEM COM DUCKDB
# ==========================================
start_duckdb = time.time()

# O DuckDB varre o arquivo no disco, não carrega tudo na RAM,
# e executa a agregação diretamente em C++ otimizado usando SQL.
query = f"""
    SELECT 
        CAST(tpep_pickup_datetime AS DATE) AS data_corrida,
        COUNT(*) AS total_corridas,
        AVG(fare_amount) AS ticket_medio
    FROM '{file_path}'
    WHERE tpep_pickup_datetime >= '2026-04-01' 
      AND tpep_pickup_datetime < '2026-05-01'
    GROUP BY 1
"""
resultado_duckdb = duckdb.query(query).df()

tempo_duckdb = time.time() - start_duckdb
print(f"Tempo DuckDB: {tempo_duckdb:.4f} segundos")

# ==========================================
print("\n--- RESULTADO ---")
if tempo_duckdb < tempo_pandas:
    print(f"DuckDB foi {tempo_pandas / tempo_duckdb:.1f}x mais rápido!")
else:
    print("Pandas foi mais rápido neste dataset pequeno.")

In [ ]:
resultado_pandas.head()

In [ ]:
# Cria uma conexão persistente local
con = duckdb.connect('nyc_taxi.duckdb')

In [ ]:
# Checando o schema
con.execute(f"DESCRIBE SELECT * FROM '{file_path}'").df()

In [ ]:
con.execute(f"SELECT * FROM '{file_path}' limit 15").df()

In [ ]:
# Limpeza e agregação: Filtrando viagens inválidas e agregando por dia
query_limpeza = f"""
    SELECT 
        tpep_pickup_datetime::DATE AS data_viagem,
        COUNT(*) AS total_viagens,
        AVG(total_amount) AS ticket_medio,
        AVG(trip_distance) AS distancia_media
    FROM '{file_path}'
    WHERE total_amount > 0 
      AND trip_distance > 0
      AND tpep_pickup_datetime >= '2026-04-01' 
      AND tpep_pickup_datetime < '2026-05-01'
    GROUP BY 1
    ORDER BY 1
"""

# Executa a query e salva em um DataFrame para visualizar
df_agregado = con.execute(query_limpeza).df()

In [ ]:
df_agregado.head()

In [ ]:
query_exportacao = f"""
    COPY ({query_limpeza}) 
    TO '../data/processed/viagens_diarias_2026_04.parquet' 
    (FORMAT PARQUET);
"""
con.execute(query_exportacao)
con.close()

In [ ]:
import pandas as pd
import plotly.express as px

# Lê o dado processado
df_final = pd.read_parquet('../data/processed/viagens_diarias_2026_04.parquet')

# Plota a tendência
fig = px.line(df_final, x='data_viagem', y='total_viagens', 
              title='Volume de Viagens de Táxi por Dia - NYC (Abril 2026)')
fig.show()
